# 03 - VWAP Backtest

Analyze VWAP (Volume-Weighted Average Price) data from the Gold layer.
Compare VWAP vs simple average price and identify divergences that
may indicate large institutional orders.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("vwap-backtest")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .getOrCreate()
)

In [ ]:
# Load Gold VWAP and OHLC for 5-minute windows
vwap_5m = spark.read.format("delta").load("s3a://cryptostream-dev-gold/vwap_5m/")
ohlc_5m = spark.read.format("delta").load("s3a://cryptostream-dev-gold/ohlc_5m/")

print(f"VWAP rows: {vwap_5m.count():,}")
print(f"OHLC rows: {ohlc_5m.count():,}")

In [ ]:
# Join VWAP with OHLC to compare
analysis = (
    vwap_5m.alias("v")
    .join(
        ohlc_5m.alias("o"),
        (F.col("v.window_start") == F.col("o.window_start")) &
        (F.col("v.symbol") == F.col("o.symbol")),
    )
    .select(
        F.col("v.window_start"),
        F.col("v.symbol"),
        F.col("v.vwap"),
        F.col("o.close").alias("close_price"),
        F.col("o.high"),
        F.col("o.low"),
        F.col("v.total_volume"),
        F.col("v.trade_count"),
    )
    .withColumn("simple_avg", (F.col("high") + F.col("low")) / 2)
    .withColumn("vwap_vs_avg_bps",
        F.round((F.col("vwap") - F.col("simple_avg")) / F.col("simple_avg") * 10000, 2)
    )
    .orderBy("window_start")
)

analysis.show(20, truncate=False)

In [ ]:
# VWAP divergence: when VWAP >> close, large buyers dominated
btc = analysis.filter(F.col("symbol") == "BTCUSDT")

divergences = (
    btc
    .filter(F.abs(F.col("vwap_vs_avg_bps")) > 10)  # > 10 bps divergence
    .orderBy(F.abs(F.col("vwap_vs_avg_bps")).desc())
)

print(f"Significant divergences (>10bps): {divergences.count()}")
divergences.show(10, truncate=False)

In [ ]:
# Volume profile: average volume and trade count by hour of day
hourly_profile = (
    vwap_5m
    .filter(F.col("symbol") == "BTCUSDT")
    .withColumn("hour", F.hour("window_start"))
    .groupBy("hour")
    .agg(
        F.avg("total_volume").alias("avg_volume"),
        F.avg("trade_count").alias("avg_trades"),
        F.count("*").alias("windows"),
    )
    .orderBy("hour")
)

hourly_profile.show(24)

In [ ]:
# Rolling VWAP comparison: 1m vs 5m vs 15m
vwap_1m = spark.read.format("delta").load("s3a://cryptostream-dev-gold/vwap_1m/")
vwap_15m = spark.read.format("delta").load("s3a://cryptostream-dev-gold/vwap_15m/")

for name, df in [("1m", vwap_1m), ("5m", vwap_5m), ("15m", vwap_15m)]:
    stats = (
        df.filter(F.col("symbol") == "BTCUSDT")
        .select(
            F.lit(name).alias("window"),
            F.count("*").alias("rows"),
            F.avg("vwap").alias("avg_vwap"),
            F.avg("total_volume").alias("avg_volume"),
            F.avg("trade_count").alias("avg_trades"),
        )
    )
    stats.show(truncate=False)

In [ ]:
spark.stop()